# 00B — PostgreSQL Performance Quickcheck
Health check rápido, read-only y reusable para `bd_replica_crm`.


In [1]:
from __future__ import annotations
import sys, time
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres

settings = load_settings(PROJECT_ROOT)
conn = connect_postgres(settings)

pd.set_option("display.max_columns",150)
pd.set_option("display.max_rows",150)
pd.set_option("display.width",220)

def df(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

print("DB:", settings.postgres.database)


DB: medallio_dw


## 1. Servidor y sesiones activas


In [2]:
display(df("""
SELECT current_database() database,current_user user_name,version(),
       now() now,pg_postmaster_start_time() postmaster_start
"""))
activity=df("""
SELECT pid,usename,application_name,state,wait_event_type,wait_event,
       now()-query_start AS running_for,LEFT(query,500) query
FROM pg_stat_activity
WHERE datname=current_database() AND pid<>pg_backend_pid()
ORDER BY query_start
""")
display(activity)


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,database,user_name,version,now,postmaster_start
0,medallio_dw,postgres,"PostgreSQL 18.4 on x86_64-windows, compiled by...",2026-09-07 04:56:13.061932+00:00,2026-09-04 01:46:28.987845+00:00


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,pid,usename,application_name,state,wait_event_type,wait_event,running_for,query
0,18876,postgres,DBeaver 26.1.1 - Main <medallio_dw>,idle,Client,ClientRead,0 days 04:24:07.829937,SHOW search_path
1,17788,postgres,DBeaver 26.1.1 - SQLEditor <Script-47.sql>,idle,Client,ClientRead,0 days 04:24:07.519017,"SET search_path = analytics,""$user"",public"
2,8568,postgres,DBeaver 26.1.1 - SQLEditor <Script-61.sql>,idle,Client,ClientRead,0 days 04:24:07.311310,"SET search_path = analytics,""$user"",public"
3,16500,postgres,DBeaver 26.1.1 - SQLEditor <Script-62.sql>,idle,Client,ClientRead,0 days 04:19:59.213367,"SELECT c.oid, a.attnum, a.attname, c.relname, ..."
4,10600,postgres,DBeaver 26.1.1 - Metadata <medallio_dw>,idle,Client,ClientRead,0 days 04:19:59.126563,"SELECT i.*,c.relnamespace FROM pg_catalog.pg_i..."
5,35548,postgres,DBeaver 26.1.1 - SQLEditor <Script-63.sql>,idle,Client,ClientRead,0 days 02:58:53.890580,"SELECT c.oid, a.attnum, a.attname, c.relname, ..."
6,34008,postgres,replica_redshift_local,active,None,None,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...
7,16924,postgres,replica_redshift_local,active,Lock,relation,0 days 00:12:22.917462,CREATE INDEX IF NOT EXISTS ix_lead_evidence_de...


## 2. Locks


In [3]:
locks=df("""
SELECT a.pid,a.usename,a.state,l.locktype,l.mode,l.granted,c.relname,
       now()-a.query_start AS running_for,LEFT(a.query,300) query
FROM pg_locks l
JOIN pg_stat_activity a ON a.pid=l.pid
LEFT JOIN pg_class c ON c.oid=l.relation
WHERE a.datname=current_database()
ORDER BY l.granted,a.query_start
""")
locks


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,pid,usename,state,locktype,mode,granted,relname,running_for,query
0,16924,postgres,active,relation,ShareLock,False,lead_evidence,0 days 00:12:22.917462,CREATE INDEX IF NOT EXISTS ix_lead_evidence_de...
1,34008,postgres,active,relation,RowExclusiveLock,True,ix_lead_evidence_project_time,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...
2,34008,postgres,active,relation,AccessShareLock,True,ix_lead_evidence_advisor_time,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...
3,34008,postgres,active,relation,RowExclusiveLock,True,ix_lead_evidence_advisor_time,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...
4,34008,postgres,active,relation,AccessShareLock,True,lead_evidence_pkey,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...
5,34008,postgres,active,relation,RowExclusiveLock,True,lead_evidence_pkey,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...
6,34008,postgres,active,relation,AccessShareLock,True,ix_lead_evidence_document_time,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...
7,34008,postgres,active,relation,RowExclusiveLock,True,ix_lead_evidence_document_time,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...
8,34008,postgres,active,relation,AccessShareLock,True,ix_lead_evidence_decision_at,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...
9,34008,postgres,active,relation,RowExclusiveLock,True,ix_lead_evidence_decision_at,0 days 00:32:43.407889,\n UPDATE features.lead_evidence e SET\...


## 3. Tablas más grandes


In [4]:
table_sizes=df("""
SELECT n.nspname schema,c.relname table,c.reltuples::bigint approx_rows,
       pg_total_relation_size(c.oid) total_bytes,
       pg_indexes_size(c.oid) index_bytes,
       pg_size_pretty(pg_total_relation_size(c.oid)) total_size
FROM pg_class c JOIN pg_namespace n ON n.oid=c.relnamespace
WHERE c.relkind='r' AND n.nspname NOT IN ('pg_catalog','information_schema')
ORDER BY total_bytes DESC LIMIT 50
""")
table_sizes


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,schema,table,approx_rows,total_bytes,index_bytes,total_size
0,raw_cygnus,interacciones,798213,335077376,26066944,320 MB
1,features,lead_evidence,205947,231628800,73195520,221 MB
2,raw_cygnus,clientes_proyectos,204102,76783616,6569984,73 MB
3,raw_cygnus,clientes,141512,73367552,4505600,70 MB
4,raw_cygnus,proforma_unidad,44100,17940480,1048576,17 MB
5,raw_cygnus,proformas,37891,11878400,892928,11 MB
6,analytics,fact_absorcion_proyecto_diario,22812,8314880,1343488,8120 kB
7,raw_cygnus,procesos,5146,6094848,524288,5952 kB
8,raw_cygnus,datos_extras,22429,4505600,1138688,4400 kB
9,raw_cygnus,unidades,3237,4349952,245760,4248 kB


## 4. Sequential scans / index scans


In [5]:
scan_stats=df("""
SELECT schemaname schema,relname table,seq_scan,seq_tup_read,idx_scan,idx_tup_fetch,
       n_live_tup,n_dead_tup,
       CASE WHEN COALESCE(seq_scan,0)+COALESCE(idx_scan,0)=0 THEN NULL
            ELSE seq_scan::numeric/(seq_scan+idx_scan) END seq_scan_share
FROM pg_stat_user_tables
ORDER BY seq_tup_read DESC NULLS LAST
LIMIT 100
""")
scan_stats


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,schema,table,seq_scan,seq_tup_read,idx_scan,idx_tup_fetch,n_live_tup,n_dead_tup,seq_scan_share
0,analytics,int_ciclo_comercial_unidad,2884324,5785084390,8.0,0.000000e+00,2007,0,0.999997
1,raw_cygnus,procesos,16169,82792656,112895.0,1.067344e+08,5148,478,0.125279
2,raw_cygnus,interacciones,118,30535625,71952.0,7.909800e+04,2525,32005,0.001637
3,raw_cygnus,clientes_proyectos,125,9653831,13641.0,1.292400e+04,717,10423,0.009080
4,raw_cygnus,clientes,40,5372252,35834.0,3.529000e+04,141559,22589,0.001115
5,features,lead_evidence,56,4698772,1162275.0,3.245070e+09,205947,0,0.000048
6,raw_cygnus,datos_extras,215,4685015,4770.0,4.651000e+03,22429,2857,0.043129
7,raw_cygnus,proforma_unidad,133,4474174,7835.0,7.415000e+03,44232,8390,0.016692
8,raw_cygnus,proformas,40,1438286,6914.0,6.547000e+03,38008,6314,0.005752
9,raw_cygnus,unidades,203,449070,3407.0,2.699600e+04,3237,249,0.056233


## 5. Vacuum / Analyze


In [6]:
maintenance=df("""
SELECT schemaname schema,relname table,n_live_tup,n_dead_tup,
       CASE WHEN n_live_tup>0 THEN n_dead_tup::numeric/n_live_tup END dead_ratio,
       last_vacuum,last_autovacuum,last_analyze,last_autoanalyze
FROM pg_stat_user_tables
ORDER BY n_dead_tup DESC LIMIT 100
""")
maintenance


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,schema,table,n_live_tup,n_dead_tup,dead_ratio,last_vacuum,last_autovacuum,last_analyze,last_autoanalyze
0,raw_cygnus,interacciones,2525,32005,12.675248,None,NaT,None,NaT
1,raw_cygnus,clientes,141559,22589,0.159573,None,NaT,None,2026-09-06 22:36:15.779475+00:00
2,raw_cygnus,clientes_proyectos,717,10423,14.536960,None,NaT,None,NaT
3,raw_cygnus,proforma_unidad,44232,8390,0.189682,None,NaT,None,2026-09-06 19:59:25.260605+00:00
4,raw_cygnus,proformas,38008,6314,0.166123,None,NaT,None,2026-09-06 19:58:24.970108+00:00
5,raw_cygnus,datos_extras,22429,2857,0.127380,None,NaT,None,2026-09-07 04:36:50.354120+00:00
6,raw_cygnus,procesos,5148,478,0.092852,None,NaT,None,2026-09-06 22:35:13.603469+00:00
7,raw_cygnus,unidades,3237,249,0.076923,None,NaT,None,2026-09-07 02:35:23.834407+00:00
8,platform_control,readiness_controls,23,63,2.739130,None,2026-09-07 04:55:50.032424+00:00,None,2026-09-07 04:32:49.519290+00:00
9,etl_control,sync_state,9,36,4.000000,None,NaT,None,2026-09-07 00:41:21.991485+00:00


## 6. Cache hit


In [7]:
cache=df("""
SELECT SUM(heap_blks_read) heap_read,SUM(heap_blks_hit) heap_hit,
       CASE WHEN SUM(heap_blks_hit)+SUM(heap_blks_read)=0 THEN NULL
            ELSE SUM(heap_blks_hit)::numeric/(SUM(heap_blks_hit)+SUM(heap_blks_read)) END cache_hit_ratio
FROM pg_statio_user_tables
""")
cache


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,heap_read,heap_hit,cache_hit_ratio
0,2473868.0,1.664685e+09,0.998516


## 7. Índices usados / no usados


In [8]:
index_usage=df("""
SELECT s.schemaname schema,s.relname table,s.indexrelname index_name,s.idx_scan,
       pg_relation_size(i.indexrelid) index_bytes,
       pg_size_pretty(pg_relation_size(i.indexrelid)) index_size
FROM pg_stat_user_indexes s
JOIN pg_index i ON i.indexrelid=s.indexrelid
ORDER BY s.idx_scan ASC,index_bytes DESC
LIMIT 100
""")
index_usage


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,schema,table,index_name,idx_scan,index_bytes,index_size
0,features,lead_evidence,ix_lead_evidence_advisor_time,0,12886016,12 MB
1,observability,quality_checks,ix_quality_checks_asset_time,0,811008,792 kB
2,analytics,fact_movimientos_stock,fact_movimientos_stock_pkey,0,548864,536 kB
3,observability,quality_checks,ix_quality_checks_status_time,0,516096,504 kB
4,observability,quality_checks,quality_checks_pkey,0,221184,216 kB
5,observability,asset_snapshots,ix_asset_snapshots_asset_time,0,196608,192 kB
6,etl_control,sync_runs,ix_sync_runs_started_at,0,106496,104 kB
7,observability,asset_snapshots,ix_asset_snapshots_time,0,106496,104 kB
8,core,dim_unidad,dim_unidad_pkey,0,106496,104 kB
9,analytics,fact_ventas_detalle,fact_ventas_detalle_pkey,0,90112,88 kB


## 8. Foco lead scoring


In [9]:
lead_focus=df("""
SELECT n.nspname schema,c.relname table,c.reltuples::bigint approx_rows,
       pg_size_pretty(pg_total_relation_size(c.oid)) total_size
FROM pg_class c JOIN pg_namespace n ON n.oid=c.relnamespace
WHERE (n.nspname='features' AND c.relname='lead_evidence')
   OR (n.nspname='core' AND c.relname='fact_ciclo_comercial_unidad')
   OR (n.nspname='decision_intelligence' AND c.relname='lead_scores')
ORDER BY pg_total_relation_size(c.oid) DESC
""")
display(lead_focus)
lead_indexes=df("""
SELECT schemaname,tablename,indexname,indexdef
FROM pg_indexes
WHERE (schemaname='features' AND tablename='lead_evidence')
   OR (schemaname='core' AND tablename='fact_ciclo_comercial_unidad')
   OR (schemaname='decision_intelligence' AND tablename='lead_scores')
ORDER BY schemaname,tablename,indexname
""")
lead_indexes


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,schema,table,approx_rows,total_size
0,features,lead_evidence,205947,221 MB
1,decision_intelligence,lead_scores,-1,32 kB
2,core,fact_ciclo_comercial_unidad,-1,0 bytes


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\3726039764.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, conn, params=params)


,schemaname,tablename,indexname,indexdef
0,decision_intelligence,lead_scores,ix_lead_scores_time_rank,CREATE INDEX ix_lead_scores_time_rank ON decis...
1,decision_intelligence,lead_scores,lead_scores_evidence_key_model_run_id_key,CREATE UNIQUE INDEX lead_scores_evidence_key_m...
2,decision_intelligence,lead_scores,lead_scores_pkey,CREATE UNIQUE INDEX lead_scores_pkey ON decisi...
3,features,lead_evidence,ix_lead_evidence_advisor_time,CREATE INDEX ix_lead_evidence_advisor_time ON ...
4,features,lead_evidence,ix_lead_evidence_decision_at,CREATE INDEX ix_lead_evidence_decision_at ON f...
5,features,lead_evidence,ix_lead_evidence_document_time,CREATE INDEX ix_lead_evidence_document_time ON...
6,features,lead_evidence,ix_lead_evidence_project_time,CREATE INDEX ix_lead_evidence_project_time ON ...
7,features,lead_evidence,lead_evidence_pkey,CREATE UNIQUE INDEX lead_evidence_pkey ON feat...


## 9. Diagnóstico automático


In [10]:
issues=[]
def add(sev,area,msg): issues.append({"severity":sev,"area":area,"message":msg})

longq=activity[pd.to_timedelta(activity["running_for"],errors="coerce")>pd.Timedelta(seconds=10)]
if len(longq): add("WARNING","queries",f"{len(longq)} query(s) >10s")
if len(locks[locks["granted"].eq(False)]): add("CRITICAL","locks","Hay locks no concedidos")
chr_=float(cache.iloc[0]["cache_hit_ratio"]) if pd.notna(cache.iloc[0]["cache_hit_ratio"]) else np.nan
if pd.notna(chr_) and chr_<0.95: add("WARNING","cache",f"Cache hit {chr_:.1%}")
high_seq=scan_stats[(scan_stats["seq_scan_share"].fillna(0)>.80)&(scan_stats["seq_tup_read"].fillna(0)>100000)]
if len(high_seq): add("WARNING","sequential_scans",f"{len(high_seq)} tabla(s) con alto sequential scan")
dead=maintenance[maintenance["dead_ratio"].fillna(0)>.20]
if len(dead): add("WARNING","vacuum",f"{len(dead)} tabla(s) con dead tuples >20%")
if not issues: add("OK","overall","Sin alertas fuertes")
diagnostic=pd.DataFrame(issues)
diagnostic


C:\Users\dinat\AppData\Local\Temp\ipykernel_27440\912920663.py:4: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  longq=activity[pd.to_timedelta(activity["running_for"],errors="coerce")>pd.Timedelta(seconds=10)]


,severity,area,message
0,WARNING,queries,8 query(s) >10s
1,CRITICAL,locks,Hay locks no concedidos
2,WARNING,sequential_scans,2 tabla(s) con alto sequential scan
3,WARNING,vacuum,7 tabla(s) con dead tuples >20%


## 10. Top candidatos a optimización


In [11]:
candidates=scan_stats.merge(table_sizes[["schema","table","total_bytes","total_size"]],on=["schema","table"],how="left")
candidates["priority_score"]=candidates["seq_tup_read"].fillna(0)*candidates["seq_scan_share"].fillna(0)
candidates.sort_values("priority_score",ascending=False).head(30)


,schema,table,seq_scan,seq_tup_read,idx_scan,idx_tup_fetch,n_live_tup,n_dead_tup,seq_scan_share,total_bytes,total_size,priority_score
0,analytics,int_ciclo_comercial_unidad,2884324,5785084390,8.0,0.000000e+00,2007,0,0.999997,540672.0,528 kB,5.785068e+09
1,raw_cygnus,procesos,16169,82792656,112895.0,1.067344e+08,5148,478,0.125279,6094848.0,5952 kB,1.037218e+07
10,analytics,fact_movimientos_stock,89,397784,0.0,0.000000e+00,6530,0,1.000000,2048000.0,2000 kB,3.977840e+05
6,raw_cygnus,datos_extras,215,4685015,4770.0,4.651000e+03,22429,2857,0.043129,4505600.0,4400 kB,2.020618e+05
3,raw_cygnus,clientes_proyectos,125,9653831,13641.0,1.292400e+04,717,10423,0.009080,76783616.0,73 MB,8.766010e+04
7,raw_cygnus,proforma_unidad,133,4474174,7835.0,7.415000e+03,44232,8390,0.016692,17940480.0,17 MB,7.468187e+04
2,raw_cygnus,interacciones,118,30535625,71952.0,7.909800e+04,2525,32005,0.001637,335077376.0,320 MB,4.999589e+04
9,raw_cygnus,unidades,203,449070,3407.0,2.699600e+04,3237,249,0.056233,4349952.0,4248 kB,2.525241e+04
14,observability,quality_checks,1,8944,0.0,0.000000e+00,8980,0,1.000000,3342336.0,3264 kB,8.944000e+03
8,raw_cygnus,proformas,40,1438286,6914.0,6.547000e+03,38008,6314,0.005752,11878400.0,11 MB,8.273144e+03


In [12]:
conn.close(); print("Conexión cerrada.")


Conexión cerrada.
